In [ ]:
!pip install -q \
transformers \
accelerate \
bitsandbytes \
torch \
sentencepiece \
fastapi \
uvicorn \
pyngrok \
nest_asyncio \
pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [ ]:
import json
import torch
import nest_asyncio

from fastapi import FastAPI
from pydantic import BaseModel

from pyngrok import ngrok

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
)
model.eval()

print("Model Loaded Successfully")

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model Loaded Successfully


In [ ]:
print(model.device)
next(model.parameters()).device

cuda:0


device(type='cuda', index=0)

In [ ]:
messages = [
        {
            "role":"system",
            "content":"You are a helpful assistant."
        },
        {
            "role":"user",
            "content":'Explain me sql like a 9 year old'
        }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    temperature=0
)

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print('response',response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Step 1
Step 2
Step 3


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Step 4
response Sure! Imagine you have a big toy box full of different toys. Each toy has its own special name, color, and size. Now, SQL is like a magical language that helps you find the exact toy you want from your toy box.
When you use SQL, you tell it what kind of toy you're looking for by giving it clues. For example, if you want to find all the red cars in your toy box, you might say something like "Find me all the toys that are red and shaped like cars."
SQL uses special words and symbols to give these clues, but the basic idea is the same as finding your favorite toy. You're telling the computer exactly what you want, and it helps you get it quickly and easily!


In [ ]:
def generate_chat(messages, temperature=0.0, max_new_tokens=512):

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    do_sample = temperature > 0.0
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
    }
    if do_sample:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
    else:
        gen_kwargs["do_sample"] = False

    outputs = model.generate(
        **inputs,
        **gen_kwargs
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
class ChatMessage(BaseModel):
    role: str
    content: str


class ChatCompletionRequest(BaseModel):
    messages: list[ChatMessage]
    temperature: float = 0.0
    max_new_tokens: int = 512


class ChatCompletionResponse(BaseModel):
    success: bool
    response: str
    error_message: str

In [ ]:
app = FastAPI(
    title="Local AI API",
    version="1.0"
)

In [ ]:
@app.get("/")
def root():
    return {
        "status":"running",
        "model":MODEL_NAME
    }

In [ ]:
@app.post("/generate")
def generate(req: ChatCompletionRequest):

    try:

        result = generate_chat(
            messages=[{"role": m.role, "content": m.content} for m in req.messages],
            temperature=req.temperature,
            max_new_tokens=req.max_new_tokens
        )

        return ChatCompletionResponse(
            success=True,
            response=result,
            error_message=""
        )

    except Exception as e:

        return ChatCompletionResponse(
            success=False,
            response="",
            error_message=str(e)
        )

In [ ]:
import os

NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    env_paths = [".env", "../.env", "Backed/.env", "../Backed/.env"]
    for env_path in env_paths:
        if os.path.exists(env_path):
            try:
                with open(env_path, "r") as f:
                    for line in f:
                        if line.strip().startswith("NGROK_AUTH_TOKEN="):
                            NGROK_AUTH_TOKEN = line.strip().split("=", 1)[1].strip().strip('"').strip("'")
                            break
            except Exception as e:
                print(f"Could not read {env_path}: {e}")
            if NGROK_AUTH_TOKEN:
                break

if NGROK_AUTH_TOKEN:
    print("Loaded NGROK_AUTH_TOKEN successfully.")
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("Warning: NGROK_AUTH_TOKEN not found in environment or .env files. Please set it manually.")

In [ ]:
import uvicorn
from threading import Thread

nest_asyncio.apply()

public_url = ngrok.connect(8000)

print(public_url)

Thread(
    target=lambda: uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    ),
    daemon=True
).start()

NgrokTunnel: "https://a96b-34-24-53-59.ngrok-free.app" -> "http://localhost:8000"


In [ ]:
print('hello world')

hello world
